In [ ]:
# importing needed requerments:

import pandas as pd
import pandas_datareader as web
import numpy as np
import math , datetime
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense , Dropout , LSTM
from tensorflow.keras.callbacks import EarlyStopping
from sklearn import preprocessing
import matplotlib.pyplot as plt
from tensorflow.keras import initializers
pd.options.mode.chained_assignment = None
import re , os
import nltk
import gensim
from gensim.models import word2vec
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import yfinance as yf
import random
from gensim.models import Word2Vec
from numpy.random import seed


In [ ]:
!pip install yfinance

In [ ]:
# word to vec method :
# Skip Gram
# CBOW (Continuous Bag of Words)
'''
copy from paper!

'''

In [ ]:
# set parameters
# start date and end date

start = datetime.datetime(2012, 1, 1)
end = datetime.datetime(2020, 1, 1)


# initialize random seed
random_seed_num = 83

# percentage of split data: 0.6 --> 60% of data used to train model and 40% of data used for test
# data for train are belong to 1 Jan 2012 to 1 Jan 2017
# data for train are belong to 1 Jan 2018 to 1 Jan 2020
data_for_train = 0.75



In [ ]:
print(start)
print(end)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# read news dataset ...
# this dataset contain ..... news

df_news = pd.read_csv('/content/drive/MyDrive/us_equities_news_dataset_shrinked.csv')
ticker = list(df_news['ticker'])
title = list(df_news['title'])
date_news = list(df_news['release_date'])

In [ ]:
# spliting words ...

corpus = []
for col in title:
        word_list = col.split(" ")
        corpus.append(word_list)

In [ ]:
def set_seed(seed=0):
    tf.random.set_seed(seed)

    np.random.seed(seed)

    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
# word to vec model train ...
# each word convert to vector---> each word present in 4d space ...

#set_seed()
#model = word2vec.Word2Vec(corpus, window=14, min_count=2, workers=1)

In [ ]:
#model.save('/content/drive/MyDrive/model_Word2Vec.model')
model = Word2Vec.load('/content/drive/MyDrive/model_word2vec.model')

In [ ]:
# function to convert word to vec ....
# max word of each title is 12 ...

def f(s):
    l = []
    word_list = s.split(" ")
    for i in word_list :
        try :
            a = model.wv[i]
            for j in a:
                l.append(j)
        except:
            continue

    if len(l) >= 48 :
        l = l[:48]

    if len(l) < 48 :
        while len(l) < 48 :
            l.append(0)

    return np.array(l)






In [ ]:


f('google says I want to paint cats.')

In [ ]:

# generating Auto_Encoder X for model training ...

set_seed(random_seed_num)
auto_encoder_x = []
for i in range(len(title)) :
    auto_encoder_x.append(f(title[i]))
auto_encoder_x = np.array(auto_encoder_x)


# building A_E model

set_seed(random_seed_num)
autoencoder = Sequential()

autoencoder.add(Dense(10, activation = "linear", input_shape=(auto_encoder_x.shape[1], ) ))

autoencoder.add(Dense(5, activation = "tanh"))

autoencoder.add(Dense(1, activation = "tanh" , name = 'code'))

autoencoder.add(Dense(5, activation = "linear"))

autoencoder.add(Dense(10, activation = "linear"))

autoencoder.add(Dense(auto_encoder_x.shape[1], activation = "linear"))

# building encoder
encoder = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('code').output)


# compile and train auto encoder
autoencoder.compile(optimizer='adam' , loss='mse')

autoencoder.fit(x=auto_encoder_x , y=auto_encoder_x , batch_size = 256 , epochs=20)



tickers = ['IBM']

#@title Default title text
main_ticker_list = []





for name_ticker in tickers:

#    try :

    historical_data_of_ticker = []
    historical_return_of_ticker = []
    buy_and_hold_of_of_ticker_portfolio_return = []

    algorithm_false_news_return_daily = []
    algorithm_true_news_return_daily = []

    algorithm_false_news_return_portfolio = []
    algorithm_true_news_return_portfolio = []



    df = yf.download(name_ticker, start='2012-01-01', end='2020-01-01')
    df = df.dropna()

    a = list(df.index)
    #print(a)

    #merge news & price...


    news_l = []

    for i in range(len(a)):
        news_l.append('0')

    for i in range(len(ticker)):
        if ticker[i] == name_ticker :
            k = 0
            for j in range(len(a)) :
                try :
                    if a[j].year == int(date_news[i].split('/')[2]) and a[j].month == int(date_news[i].split('/')[0]) and a[j].day == int(date_news[i].split('/')[1]) :
                        news_l[j] = title[i]
                except :
                  continue


    df['news'] = news_l


    # generate news concept prediction ...

    news = list(df['news'])

    news_concpt = []

    for i in news :
        news_concpt.append( encoder.predict(f(i).reshape(1 , f(i).shape[0])) [0][0] )

    news_concpt = np.array(news_concpt)




    # generate daily return

    closing = np.array(df['Adj Close'])

    historical_data_of_ticker = closing

    return_ = (closing[1:] - closing[:-1]) / closing[:-1]

    historical_return_of_ticker = return_




    # generate daily scaled_log_return

    closing_log = np.log(np.array(df['Adj Close']))
    return_log = closing[1:] - closing[:-1]
    return_log_scale = preprocessing.scale(return_)





    # X & Y to train model



    x_news_f = []
    y_news_f = []
    for i in range(10 , len(closing)-1):
        x_news_f.append(return_log_scale[i-10:i])
        y_news_f.append(return_[i])
    x_news_f = np.reshape(np.array(x_news_f) , (len(x_news_f) , 10 , 1))
    y_news_f = np.array(y_news_f)



    x_news_t = []
    y_news_t = []
    for i in range(10 , len(closing)-1):

        step_t = []
        step_t.append(return_log_scale[i-10:i])
        step_t.append(news_concpt[i-10:i])
        step_t = np.array(step_t)
        step_t = step_t.transpose()

        x_news_t.append(step_t)
        y_news_t.append(return_[i])
    x_news_t = np.array(x_news_t)
    y_news_t = np.array(y_news_t)




    # number of sample in train
    train_sample_count = int(x_news_t.shape[0] * data_for_train)



    #############################. new reshape code! diffrent LSTM & ANN ###################
    # x_news_f = x_news_f.reshape((len(x_news_f) , x_news_f.shape[2] * x_news_f.shape[3]))
    # x_news_t = x_news_t.reshape((len(x_news_t) , x_news_t.shape[2] * x_news_t.shape[3]))


    #############################. new reshape code! diffrent LSTM & ANN ###################

    v_new_return_using_news_true = 0
    v_new_return_using_news_false = 0
    v_new_return_buy_hold = 0


    # build model
    ## new_TRUE

    set_seed(random_seed_num)
    model_f_news_true = Sequential()
    model_f_news_true.add(LSTM(20, activation='tanh' , return_sequences=True , input_shape=(10, x_news_t.shape[2] )))
    model_f_news_true.add(LSTM(10, activation='tanh'))
    model_f_news_true.add(Dense(8 , activation='tanh'))
    model_f_news_true.add(Dense(4 , activation='tanh'))
    model_f_news_true.add(Dense(1))
    model_f_news_true.compile(optimizer='adam', loss='mse')

    # train model
    # model_f_news_true.fit(x_news_t[:train_sample_count , : , : ], y_news_t[:train_sample_count] , batch_size=32  , epochs=10)
    model_f_news_true.fit(x_news_t[:train_sample_count , : , : ], y_news_t[:train_sample_count] , validation_data=(x_news_t[train_sample_count: , : , : ], y_news_t[train_sample_count:]) , batch_size=32  , epochs=10)


    # build model
    ## new_FALSE
    set_seed(random_seed_num)
    model_f_news_false = Sequential()
    model_f_news_false.add(LSTM(20, activation='tanh' , return_sequences=True , input_shape=(10, x_news_f.shape[2] )))
    model_f_news_false.add(LSTM(10, activation='tanh'))
    model_f_news_false.add(Dense(8 , activation='tanh'))
    model_f_news_false.add(Dense(4 , activation='tanh'))
    model_f_news_false.add(Dense(1))
    model_f_news_false.compile(optimizer='adam', loss='mse')

    # train model
    #model_f_news_false.fit(x_news_f[:train_sample_count , : , :], y_news_f[:train_sample_count] ,validationdata batch_size=32  , epochs=10)
    model_f_news_false.fit(x_news_f[:train_sample_count , : , : ], y_news_f[:train_sample_count] , validation_data=(x_news_f[train_sample_count: , : , : ], y_news_f[train_sample_count:]) , batch_size=32  , epochs=10)




    # prediction news TRUE
    # predict x_test using model
    prediction_news_true = model_f_news_true.predict(x_news_t[train_sample_count:,:,:])



    # prediction news FALSE
    # predict x_test using model
    prediction_news_false = model_f_news_false.predict(x_news_f[train_sample_count:,:,:])






    ###############################################################################################################
    ###############################################################################################################




    # news false :


    algorithm_return = []
    stock_return = []


    var_long_pos_number_new_true = 0
    var_long_pos_number_new_false = 0

    var_short_pos_number_new_true = 0
    var_short_pos_number_new_false = 0

    var_postitive_day_new_true = 0
    var_negative_day_new_true = 0

    var_postitive_day_new_false = 0
    var_negative_day_new_false = 0






    k = []

    for i in range(len(prediction_news_false)):
        if prediction_news_false[i] > 0:
            k.append(y_news_f[train_sample_count+i])
        else:
            k.append(-y_news_f[train_sample_count+i])

        if prediction_news_false[i] > 0 and y_news_f[train_sample_count+i] > 0 :
            var_long_pos_number_new_false = var_long_pos_number_new_false + 1
            var_postitive_day_new_false = var_postitive_day_new_false + 1

        if prediction_news_false[i] < 0 and y_news_f[train_sample_count+i] > 0 :
            var_short_pos_number_new_false = var_short_pos_number_new_false + 1
            var_negative_day_new_false = var_negative_day_new_false + 1

        if prediction_news_false[i] > 0 and y_news_f[train_sample_count+i] < 0 :
            var_long_pos_number_new_false = var_long_pos_number_new_false + 1
            var_negative_day_new_false = var_negative_day_new_false + 1

        if prediction_news_false[i] < 0 and y_news_f[train_sample_count+i] < 0 :
            var_short_pos_number_new_false = var_short_pos_number_new_false + 1
            var_postitive_day_new_false = var_postitive_day_new_false + 1





    algorithm_false_news_return_daily = k



    buy_and_hold = 1
    for i in y_news_f[train_sample_count : ] :
        buy_and_hold = buy_and_hold * (1+i)
        stock_return.append(buy_and_hold)

    buy_and_hold_of_of_ticker_portfolio_return = stock_return


    portfolio = 1
    for i in k :
        portfolio = portfolio * (1+i)
        algorithm_return.append(portfolio)



    algorithm_false_news_return_portfolio = algorithm_return


    print(portfolio , buy_and_hold)


    v_new_return_using_news_false = portfolio
    v_new_return_buy_hold = buy_and_hold





    plt.plot(stock_return)
    plt.plot(algorithm_return)
    plt.show()




    ##################################################################




    # news true :


    algorithm_return = []
    stock_return = []


    k = []

    number_of_buy_sell = []

    for i in range(len(prediction_news_true)):
        if prediction_news_true[i] > 0:
            number_of_buy_sell.append(1)
            k.append(y_news_t[train_sample_count+i])
        else:
            k.append(-y_news_t[train_sample_count+i])
            number_of_buy_sell.append(-1)




        if prediction_news_true[i] > 0 and y_news_t[train_sample_count+i] > 0 :
            var_long_pos_number_new_true = var_long_pos_number_new_true + 1
            var_postitive_day_new_true = var_postitive_day_new_true + 1

        if prediction_news_true[i] < 0 and y_news_t[train_sample_count+i] > 0 :
            var_short_pos_number_new_true = var_short_pos_number_new_true + 1
            var_negative_day_new_true = var_negative_day_new_true + 1

        if prediction_news_true[i] > 0 and y_news_t[train_sample_count+i] < 0 :
            var_long_pos_number_new_true = var_long_pos_number_new_true + 1
            var_negative_day_new_true = var_negative_day_new_true + 1

        if prediction_news_true[i] < 0 and y_news_t[train_sample_count+i] < 0 :
            var_short_pos_number_new_true = var_short_pos_number_new_true + 1
            var_postitive_day_new_true = var_postitive_day_new_true + 1








    algorithm_true_news_return_daily = k




    buy_and_hold = 1
    for i in y_news_f[train_sample_count : ] :
        buy_and_hold = buy_and_hold * (1+i)
        stock_return.append(buy_and_hold)




    portfolio = 1
    for i in k :
        portfolio = portfolio * (1+i)
        algorithm_return.append(portfolio)


    algorithm_true_news_return_portfolio = algorithm_return



    print(portfolio , buy_and_hold)


    v_new_return_using_news_true = portfolio





    (pd.DataFrame(buy_and_hold_of_of_ticker_portfolio_return)).to_csv( '/content/drive/MyDrive/File Storage/buy_and_hold_of_of_ticker_portfolio_return' + name_ticker + '.csv')
    (pd.DataFrame(algorithm_false_news_return_daily)).to_csv('/content/drive/MyDrive/File Storage/algorithm_false_news_return_daily' + name_ticker + '.csv')
    (pd.DataFrame(algorithm_true_news_return_daily)).to_csv('/content/drive/MyDrive/File Storage/algorithm_true_news_return_daily' + name_ticker + '.csv')
    (pd.DataFrame(algorithm_false_news_return_portfolio)).to_csv('/content/drive/MyDrive/File Storage/algorithm_false_news_return_portfolio' + name_ticker + '.csv')
    (pd.DataFrame(algorithm_true_news_return_portfolio)).to_csv('/content/drive/MyDrive/File Storage/algorithm_true_news_return_portfolio' + name_ticker + '.csv')


    (pd.DataFrame(historical_data_of_ticker)).to_csv('/content/drive/MyDrive/File Storage/historical_data_of_ticker' + name_ticker + '.csv')
    (pd.DataFrame(historical_return_of_ticker)).to_csv('/content/drive/MyDrive/File Storage/historical_return_of_ticker' + name_ticker + '.csv')




    var_list = [ ['var_long_pos_number_new_true' , var_long_pos_number_new_true ] ,
    ['var_long_pos_number_new_false' , var_long_pos_number_new_false ] ,

    ['var_short_pos_number_new_true' , var_short_pos_number_new_true ] ,
    ['var_short_pos_number_new_false' , var_short_pos_number_new_false ] ,

    ['var_postitive_day_new_true' , var_postitive_day_new_true ] ,
    ['var_negative_day_new_true' , var_negative_day_new_true ] ,

    ['var_postitive_day_new_false' , var_postitive_day_new_false ] ,
    ['var_negative_day_new_false' , var_negative_day_new_false ] ]

    (pd.DataFrame(var_list)).to_csv('/content/drive/MyDrive/File Storage/long_short_pos_neg_' + name_ticker + '.csv')









    print(' buy and hold return : ' , v_new_return_buy_hold , ' news false return : ' ,  v_new_return_using_news_false , ' news true return : ' ,  v_new_return_using_news_true, name_ticker )




    plt.plot(stock_return)
    plt.plot(algorithm_return)
    plt.show()






#    except :
#        continue

In [ ]:
83
['AAPL','GOOGL','BLK','CME','CVX','AA','CMG','JNJ','PFE','ADP','JWN','BHC','AVGO','NLOK','HUM','MSFT','BAC','AMZN','NWSA','BA','GS','TSLA','NFLX','TGT','INTC','DIS','XOM','JPM','MS','GM','C','GE','MU','TM','WMB','KO','WFC','WMT','IBM','UBER','SO','VZ','LMT','EFX','CAT','BKR','SBUX','SLB','T','NKE','PGR','AMGN','ICE','FCX','HD','FDX','DAL','PG','MRK','CAJ']
['AAPL','GOOGL','BLK','CME','CVX','AA','CMG','JNJ','PFE','ADP','JWN','BHC','AVGO','NLOK','HUM']
buy and hold return :  1.7565870199634164  news false return :  1.8701856559470007  news true return :  2.1218657904395197+AAPL
buy and hold return :  1.227086989410999  news false return :  1.2834442095396221  news true return :  1.3565219809618592+GOOGL
buy and hold return :  1.0325497496648552  news false return :  1.0983521540058498  news true return :  1.3053127698184885+BLK
buy and hold return :  1.4387437321534156  news false return :  1.5572727712217462  news true return :  1.6434130661469482+CME
buy and hold return :  1.013496898709798  news false return :  1.1826057673413797  news true return :  1.4003910558701098+CVX
buy and hold return :  0.3946788793729583  news false return :  1.1150096713483826  news true return :  2.0505836458406907+AA
buy and hold return :  2.709093803726734  news false return :  2.835377261953854  news true return :  2.835377261953854 +CMG
buy and hold return :  1.0959732230143493  news false return :  1.1650180113759911  news true return :  1.1971755764976582+JNJ
buy and hold return :  1.146455469922387  news false return :  1.3302238702413511  news true return :  1.2747722005056625-PFE`
buy and hold return :  1.5154734197162392  news false return :  1.4909161174805061  news true return :  1.5199021448323522-ADP
buy and hold return :  0.9024667469029471  news false return :  0.8972947493688205  news true return :  1.09536301185821-JWN*
buy and hold return :  1.3003042059399355  news false return :  1.1965908526229727  news true return :  1.5781245095773875-BHC*
buy and hold return :  1.2727065012067473  news false return :  1.4201496820457105  news true return :  1.367250964244353-AVGO```
buy and hold return :  0.9086017037294037  news false return :  0.7350222171045718  news true return :  0.7357200247790308-NLOK
buy and hold return :  1.4800238066074742  news false return :  0.48119023801883987  news true return :  0.5294529822107716-HUM
buy and hold return :  1.8853744198667601  news false return :  1.7120131480944463  news true return :  2.025734490117798 MSFT-*
buy and hold return :  1.2309910066949292  news false return :  1.33196713858202  news true return :  1.2512707855633598 BAC-``
buy and hold return :  1.5344959258576423  news false return :  1.2364358908073996  news true return :  1.1787536480736818 AMZN-
buy and hold return :  0.9102415064419875  news false return :  1.1114112658844293  news true return :  0.8354358991299433 NWSA-
buy and hold return :  1.1410566045851436  news false return :  0.8564946150139967  news true return :  0.8999453584338877 BA-
buy and hold return :  0.9392504316392847  news false return :  0.9392504316392847  news true return :  0.9154667424786169 GS-
buy and hold return :  1.3186130710825685  news false return :  0.524878533683834  news true return :  0.8315330827440258 TSLA-
buy and hold return :  1.5780053767789746  news false return :  1.5772434048044641  news true return :  1.5669040880962992 NFLX-
buy and hold return :  2.0353292184357965  news false return :  2.0991376332080716  news true return :  1.9593165403300188 TGT-
buy and hold return :  1.389456081775638  news false return :  1.2702847089207765  news true return :  1.4587308581078566 INTC-*
buy and hold return :  1.324537341589733  news false return :  1.0104176740195094  news true return :  1.029211581121396 DIS-
buy and hold return :  0.8783443180213335  news false return :  1.4766707571823052  news true return :  1.201092610359169 XOM-``
buy and hold return :  1.3607788050162568  news false return :  1.3553536245390165  news true return :  1.3607788050162568 JPM-
buy and hold return :  1.0275227737820591  news false return :  0.7912129467062534  news true return :  0.8355581578609799 MS-
buy and hold return :  0.927121554484562  news false return :  0.7383589836418784  news true return :  0.8300680017137348 GM-
buy and hold return :  1.1259090680728765  news false return :  1.088174344509679  news true return :  0.9406828238577152 C-
buy and hold return :  0.6602299488520494  news false return :  2.5098744366819665  news true return :  2.431056128323102 GE-````
buy and hold return :  1.19564250009684  news false return :  1.098603840436069  news true return :  1.502047239932242 MU-*
buy and hold return :  1.0799968340329742  news false return :  1.0799968340329742  news true return :  1.0799968340329742 TM-
 buy and hold return :  0.8281684726589215  news false return :  0.7128277275226444  news true return :  0.6297451306584075 WMB-
buy and hold return :  1.3001510721895309  news false return :  1.2588180217485498  news true return :  1.284163873119184 KO-
buy and hold return :  0.9355906249620288  news false return :  0.9377758543758247  news true return :  0.9230887621942779 WFC-
buy and hold return :  1.2481109487468682  news false return :  1.2481109487468682  news true return :  1.2481109487468682 WMT-
buy and hold return :  0.9263841775520275  news false return :  0.5733239582562712  news true return :  0.6048712591450528 IBM-
buy and hold return :  1.061384699687894  news false return :  1.0065954601684082  news true return :  0.8969035253776196 UBER-
buy and hold return :  1.4966644966923213  news false return :  1.8573912003088002  news true return :  0.9076819788552727 SO-
buy and hold return :  1.27859670355739  news false return :  1.1981479101318782  news true return :  1.528997691609383 VZ-*
buy and hold return :  1.2754793754927338  news false return :  0.997351000329764  news true return :  1.085296501573022 LMT-
buy and hold return :  1.1963966862917128  news false return :  1.1781364696324477  news true return :  1.2384122872059042 EFX-*
buy and hold return :  0.9871723850069346  news false return :  1.147115116135044  news true return :  1.0944738937017502 CAT-
buy and hold return :  0.8076180549317641  news false return :  0.7755559411410883  news true return :  0.5703260258650166 BKR-
buy and hold return :  1.561017548154488  news false return :  1.7197184980485112  news true return :  1.4751404944593123 SBUX-
buy and hold return :  0.6155354012375934  news false return :  0.46715419654470963  news true return :  0.5102181341877422 SLB-
buy and hold return :  1.214749411425161  news false return :  1.4894633426703303  news true return :  1.4064308877907479 T-
buy and hold return :  1.63105746669338  news false return :  1.9191582658511335  news true return :  1.5059938956299173 NKE-
buy and hold return :  1.3934070472015725  news false return :  1.2202003137334905  news true return :  1.2119221293439553 PGR-
buy and hold return :  1.4178784918181673  news false return :  1.3995232624088114  news true return :  1.281554279114334 AMGN-
 buy and hold return :  1.3389882544460716  news false return :  1.0212898446597694  news true return :  0.857321312625068 ICE-
 buy and hold return :  0.6927238651811778  news false return :  0.7042090126433023  news true return :  0.6955085126199959 FCX-
buy and hold return :  1.2136781267247716  news false return :  1.2136781267247716  news true return :  1.2136781267247716 HD-
buy and hold return :  0.5951483745915768  news false return :  0.8904245189661312  news true return :  0.6123148646309102 FDX-
buy and hold return :  1.1056359006118104  news false return :  0.5405898616825588  news true return :  1.5185139041306686 DAL-*
 buy and hold return :  1.4682847864688229  news false return :  1.5157391784145708  news true return :  1.501509901292777 PG-``
buy and hold return :  1.715198689921779  news false return :  1.8652150638435596  news true return :  1.8725171924212338 MRK+
buy and hold return :  0.7206851075528492  news false return :  0.7270788340204597  news true return :  0.7196432548593346 CAJ-

In [ ]:
8
['AAPL','MSFT','AMZN','XOM','WMB','WFC','AA','EFX','JWN','BHC']
'SLB','JNJ','T','NKE','PGR','AMGN','ICE','FCX','HD','FDX','AAPL','MSFT','BAC','AMZN','NWSA','BA','GOOGL','GS','TSLA','NFLX','TGT','INTC','DIS','XOM','JPM','MS','GM','C','GE','MU','TM','WMB','KO','WFC','WMT','IBM','UBER','BLK','CME','CVX','AA','CMG','SO','VZ','LMT','EFX','CAT','BKR','SBUX','DAL','UBSG','PG','SNE','MRK','CAJ','PFE','ADP','JWN','BHC','AVGO','NLOK','HUM''CAT','BKR','SBUX','DAL','MRK','CAJ','PFE','ADP','JWN','BHC','AVGO','NLOK','HUM']
buy and hold return :  0.6155354393446092  news false return :  1.151994228699257  news true return :  0.9607908624846039-SLB
buy and hold return :  1.095973112465444  news false return :  0.8139944063984574  news true return :  0.9439665691697631-
buy and hold return :  1.2147495307560467  news false return :  1.0048832666054666  news true return :  1.050622784524941-
buy and hold return :  1.6310576945949797  news false return :  1.8632351667603908  news true return :  1.5979327302251756-
buy and hold return :  1.3934069338421957  news false return :  0.9928530548464557  news true return :  1.026571266890159-
buy and hold return :  1.417878725544064  news false return :  1.3715705819392734  news true return :  1.1523859523837807-
buy and hold return :  1.3389881020791983  news false return :  0.6135268892171258  news true return :  0.7611218062467349-
buy and hold return :  0.6927238849142155  news false return :  1.100840440317255  news true return :  0.9772153332653573-
buy and hold return :  1.2136776014701744  news false return :  1.2136776014701744  news true return :  1.2136776014701744-
buy and hold return :  0.595148275948005  news false return :  0.9077701445815021  news true return :  0.6361095396693388-
buy and hold return :  1.7565866936360324  news false return :  2.222923546975538  news true return :  2.9398079195208178+AAPL
buy and hold return :  1.8853747826090197  news false return :  2.4112825207551123  news true return :  2.4656417513328432+MSFT
buy and hold return :  1.2309911477254551  news false return :  0.801131527850089  news true return :  0.8092356689250177-
buy and hold return :  1.5344959258576423  news false return :  1.6720776436688947  news true return :  1.7793641363806558+AMZN
buy and hold return :  0.9102415640035157  news false return :  0.6322904829356863  news true return :  0.6540922767899813-
buy and hold return :  1.1410566045851431  news false return :  0.7289035944403044  news true return :  0.7351990666522329-
buy and hold return :  1.227086989410999  news false return :  2.028024335811697  news true return :  1.774249083681806-
buy and hold return :  0.9392504316392853  news false return :  0.9472172009114114  news true return :  0.9392504316392853-
buy and hold return :  1.3186130710825685  news false return :  0.9800901445114362  news true return :  0.9939346564772156-
 buy and hold return :  1.5780053767789746  news false return :  1.245466991454741  news true return :  0.9243452656447291-
buy and hold return :  2.035329475545353  news false return :  0.4828748574234844  news true return :  0.46553614451114744-
buy and hold return :  1.3894562709099838  news false return :  1.4875386710276775  news true return :  1.3647281687325143-
buy and hold return :  1.3245373415897337  news false return :  0.7892787761799438  news true return :  1.0021274261180644-
buy and hold return :  0.878344318021335  news false return :  0.9087015503429721  news true return :  0.9965222263123588+XOM
buy and hold return :  1.3607790232381547  news false return :  1.0138086019516346  news true return :  1.2674478772440745-
 buy and hold return :  1.0275228559111538  news false return :  0.7946053329028708  news true return :  0.9897175016645636-
buy and hold return :  0.9271214638031526  news false return :  1.149389113580287  news true return :  0.6338632869049161-
buy and hold return :  1.1259088185285766  news false return :  1.2973763274960441  news true return :  1.1773180060073198-
buy and hold return :  0.6602297597981278  news false return :  1.2686576768417799  news true return :  1.0457534339691312-
buy and hold return :  1.1956425852308645  news false return :  1.0670091814290594  news true return :  1.022276801841673-
buy and hold return :  1.0799968340329742  news false return :  0.8104582649648914  news true return :  1.0614000062713072-
buy and hold return :  0.8281685516260867  news false return :  0.8766476755866666  news true return :  0.9269178704335026+WMB
buy and hold return :  1.3001511013106466  news false return :  0.6440432188495041  news true return :  0.6409729239369453-
buy and hold return :  0.9355905588684686  news false return :  1.0688349306886638  news true return :  1.268298142707263+WFC
buy and hold return :  1.248110636535133  news false return :  0.8698582730629616  news true return :  0.9597776976641231-
buy and hold return :  0.9263841151019611  news false return :  0.8939838548876862  news true return :  0.9250834003648623-
buy and hold return :  1.061384699687894  news false return :  1.0182703862381415  news true return :  0.9515616459708998-
buy and hold return :  1.0325498869188185  news false return :  1.1757431644879655  news true return :  1.047633563060085-
buy and hold return :  1.4387441366674796  news false return :  1.3567609424861726  news true return :  1.599978098897813-*
buy and hold return :  1.013496677680484  news false return :  0.9824016702853485  news true return :  1.0168735365922064-
buy and hold return :  0.39467885160912947  news false return :  1.9404583547857326  news true return :  1.9520675967748478+AA
buy and hold return :  2.709093803726734  news false return :  0.5315918053759807  news true return :  0.5350646677456559-
buy and hold return :  1.4966640505084814  news false return :  0.6132007076195531  news true return :  0.6132007076195531-
buy and hold return :  1.2785963877424655  news false return :  1.5839578719939063  news true return :  1.3162766427240347-
buy and hold return :  1.2754797850309634  news false return :  0.6191474889862725  news true return :  0.7294039996742231-
buy and hold return :  1.1963968192602241  news false return :  1.459937124499606  news true return :  1.5624991967259618+EFX
buy and hold return :  0.9871722780932864  news false return :  0.9646027503459029  news true return :  0.9938354565218444-*
buy and hold return :  0.8076180675179955  news false return :  0.48771699642229616  news true return :  0.5006416864857408-
buy and hold return :  1.5610175481544875  news false return :  0.7013597146485748  news true return :  0.7049377070695362-
buy and hold return :  1.1056357400294679  news false return :  1.1056357400294679  news true return :  1.1056357400294679-
buy and hold return :  1.468284741862074  news false return :  0.6670656499715091  news true return :  0.6677687053856248-
 buy and hold return :  1.715198853911146  news false return :  1.5306943009524863  news true return :  1.2999228158618572-
buy and hold return :  0.7206851075528492  news false return :  0.7548578213522943  news true return :  0.7548578213522943-
buy and hold return :  1.1464556732791764  news false return :  1.4244590658980194  news true return :  1.131023063597257-
buy and hold return :  1.5154733850857969  news false return :  1.4090639185348222  news true return :  1.4588116462857466-
buy and hold return :  0.9024669113931129  news false return :  1.0188410717897247  news true return :  0.9256433180114076-
buy and hold return :  1.3003042059399355  news false return :  0.1881752335340539  news true return :  0.27382161146047423-
buy and hold return :  1.2727062831980192  news false return :  1.4051119333711681  news true return :  1.4051119333711681-+'AVGO',
buy and hold return :  0.9086017600805679  news false return :  1.3436578022329861  news true return :  1.33942598417466-+'NLOK'
buy and hold return :  1.480024209477666  news false return :  1.480024209477666  news true return :  1.480024209477666
